# 05 工具发现 — Embedding 驱动的语义检索

工具数量从几个增长到几十上百时，全量发给 LLM 会导致 token 爆炸 + 选择困难。
05 引入 embedding 向量检索，用通用的 EmbeddingStore 同时解决**工具过滤**和**语义记忆搜索**两个问题。

**学习点**：embedding 不是 LLM 的输入格式，而是检索的"中间人"——最终传给 LLM 的永远是自然语言。

In [1]:
import sys, json, shutil, os
sys.path.insert(0, '..')

from src.agent_framework import Agent, EmbeddingStore, ToolRegistry
from src.capabilities import LongTermMemory

# 清理之前实验的记忆
for d in ['agent_memory']:
    if os.path.exists(d):
        shutil.rmtree(d)

print('一切就绪')

/home/yixienaqi0818/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


一切就绪


## EmbeddingStore 基础

多 collection 隔离：`tools`、`memories`、未来 `rag_docs` 各搜各的。

In [2]:
store = EmbeddingStore()

# 往不同 collection 添加数据
store.add("tools", "查询指定城市的天气情况", {"name": "get_weather"})
store.add("tools", "执行数学计算，加减乘除等", {"name": "calculate"})
store.add("tools", "向指定邮箱发送邮件", {"name": "send_email"})

store.add("memories", "用户叫小明，住在北京", {"timestamp": "2026-07-04"})
store.add("memories", "用户喜欢喝咖啡", {"timestamp": "2026-07-04"})
store.add("memories", "用户讨厌香菜", {"timestamp": "2026-07-04"})

print(f"Collections: {store.list_collections()}")
print(f"Total items: {len(store)}")

Loading weights: 100%|██████████| 71/71 [00:00<00:00, 1392.28it/s]


Collections: ['tools', 'memories']
Total items: 6


### 各搜各的，互不干扰

In [3]:
print("=== 搜工具 ===")
for r in store.search("tools", "今天天气怎么样", top_k=3):
    print(f"  [{r['meta']['name']}] score={r['score']:.3f}")

print("\n=== 搜记忆 ===")
for r in store.search("memories", "用户住在哪里", top_k=3):
    print(f"  score={r['score']:.3f}  {r['text']}")

=== 搜工具 ===
  [get_weather] score=0.523
  [send_email] score=0.286
  [calculate] score=0.283

=== 搜记忆 ===
  score=0.671  用户叫小明，住在北京
  score=0.509  用户喜欢喝咖啡
  score=0.452  用户讨厌香菜


### 语义相似度

跨 collection 的通用相似度计算，替代 Jaccard 字符集匹配。

In [4]:
# embedding 能识别同义表达
print(f"'程序员' vs '写代码为生': {store.similarity('程序员', '写代码为生'):.3f}")
print(f"'北京' vs '京北': {store.similarity('北京', '京北'):.3f}")
print(f"'咖啡' vs '茶叶': {store.similarity('咖啡', '茶叶'):.3f}")
print(f"'咖啡' vs '汽车': {store.similarity('咖啡', '汽车'):.3f}")

'程序员' vs '写代码为生': 0.543
'北京' vs '京北': 0.914
'咖啡' vs '茶叶': 0.498
'咖啡' vs '汽车': 0.441


## 工具过滤 vs 全量工具

模拟 10+ 个工具的场景，对比全量发送和 embedding 过滤。

In [5]:
es = EmbeddingStore()
tr = ToolRegistry(es)

tools_batch = [
    ("get_weather", "查询指定城市的天气"),
    ("search_web", "搜索互联网获取信息"),
    ("calculate", "执行数学计算"),
    ("send_email", "向指定邮箱发送邮件"),
    ("play_music", "播放指定歌曲"),
    ("translate", "将文本翻译成指定语言"),
    ("set_alarm", "设置闹钟提醒"),
    ("check_stock", "查询股票价格"),
    ("book_flight", "预订航班机票"),
    ("order_food", "在线点餐外卖"),
]

for name, desc in tools_batch:
    tr.register(name, desc,
        {"type": "object", "properties": {}, "required": []},
        lambda: "done")

print(f"注册了 {len(tr)} 个工具")

注册了 10 个工具


In [8]:
test_queries = [
    "今天北京天气如何",
    "帮我算一下 123 * 456",
    "给老板发封邮件",
    "翻译这句话成英文",
]

for query in test_queries:
    all_names = [t['function']['name'] for t in tr.get_definitions()]
    filtered = [t['function']['name'] for t in tr.get_definitions(query=query, top_k=3)]
    print(f"{query}")
    print(f"  全量({len(all_names)}): {all_names}")
    print(f"  过滤(3): {filtered}\n")

今天北京天气如何
  全量(10): ['get_weather', 'search_web', 'calculate', 'send_email', 'play_music', 'translate', 'set_alarm', 'check_stock', 'book_flight', 'order_food']
  过滤(3): ['get_weather', 'search_web', 'send_email']

帮我算一下 123 * 456
  全量(10): ['get_weather', 'search_web', 'calculate', 'send_email', 'play_music', 'translate', 'set_alarm', 'check_stock', 'book_flight', 'order_food']
  过滤(3): ['calculate', 'get_weather', 'check_stock']

给老板发封邮件
  全量(10): ['get_weather', 'search_web', 'calculate', 'send_email', 'play_music', 'translate', 'set_alarm', 'check_stock', 'book_flight', 'order_food']
  过滤(3): ['translate', 'check_stock', 'send_email']

翻译这句话成英文
  全量(10): ['get_weather', 'search_web', 'calculate', 'send_email', 'play_music', 'translate', 'set_alarm', 'check_stock', 'book_flight', 'order_food']
  过滤(3): ['translate', 'book_flight', 'send_email']



## 长期记忆：embedding 语义检索

之前用字符级 Jaccard，"程序员" 和 "写代码为生" 完全匹配不到。
现在用 embedding 余弦相似度。

In [10]:
ltm = LongTermMemory(embedding_store=EmbeddingStore())

# 添加语义相近的记忆
ltm.add("用户是程序员，主要用 Python")
ltm.add("用户住在北京朝阳区")
ltm.add("用户每天早上喝咖啡")
ltm.add("用户喜欢骑自行车上班")

print("所有记忆:")
for m in ltm.list_all():
    print(f"  [{m["index"]}] {m["content"]}")


TypeError: LongTermMemory.__init__() missing 1 required positional argument: 'embedding_store'

In [ ]:
# 语义搜索：字面不重叠但语义相关
for query in ["用户从事什么职业", "用户住在哪", "用户喜欢喝什么", "用户通勤方式"]:
    results = ltm.search(query, top_k=2)
    print(f"\nQ: {query}")
    for r in results:
        print(f"  [{r['score']:.3f}] {r['content']}")

### 相似度去重

embedding 能识别"开发"和"编程"是同义，不会重复存储。

In [ ]:
before = len(ltm.list_all())
is_new = ltm.add("用户做软件开发工作")  # 和 "用户是程序员" 语义相同
after = len(ltm.list_all())

print(f"添加前: {before}, 添加后: {after}")
print(f"是新增: {is_new}")

# 检查——应该是覆盖了第一条，而不是新增
for m in ltm.list_all():
    print(f"  [{m['index']}] {m['content']}")

## 端到端：Agent + 工具过滤

用完整的 Agent 跑一遍，确认工具过滤 + 语义记忆都能正常工作。

In [ ]:
from src.capabilities.demo_tools import create_demo_tools
from src.capabilities import PlanManager

# 共享 embedding store
es2 = EmbeddingStore()
ltm2 = LongTermMemory(embedding_store=es2)
ltm2.add("用户叫小明，喜欢简洁的回答")

plan_mgr = PlanManager()
tools = create_demo_tools(plan_mgr=plan_mgr, ltm=ltm2)

agent = Agent(
    tools=tools, long_term_memory=ltm2, plan_mgr=plan_mgr,
    tool_top_k=5, embedding_store=es2,
)

print(f"全部工具数: {len(tools)}")
print(f"过滤 top_k: {agent.tool_top_k}")
print(f"Embedding collections: {es2.list_collections()}")

In [ ]:
# 测试：简单查询，验证工具过滤和记忆
reply = agent.chat(
    "我叫什么名字？北京今天天气怎么样？"
)
print(f"\n=== 最终回答 ===\n{reply}")

## 清理

In [ ]:
for d in ['agent_memory']:
    if os.path.exists(d):
        shutil.rmtree(d)
print('清理完成')